In [2]:
#Question 2

import pandas as pd
import re
import emoji
import string
import nltk
from bs4 import BeautifulSoup
from autocorrect import Speller
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

# Download required NLTK resources [cite: 310, 311, 313, 314, 316]
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

# Initialize tools [cite: 318, 319, 320, 321]
spell = Speller(lang='en')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Dictionaries for cleaning [cite: 322, 343]
slang_dict = {
    "tbh": "to be honest", "omg": "oh my god", "lol": "laugh out loud",
    "idk": "I don't know", "brb": "be right back", "btw": "by the way",
    "imo": "in my opinion", "smh": "shaking my head", "fyi": "for your information",
    "np": "no problem", "ikr": "I know right", "asap": "as soon as possible"
}

contractions_dict = {
    "wasn't": "was not", "isn't": "is not", "aren't": "are not", "don't": "do not",
    "can't": "cannot", "i'm": "i am", "you're": "you are", "it's": "it is"
}

# --- Pre-processing Functions [cite: 392, 394, 397, 400, 428, 442, 445, 450, 453, 465, 475] ---

def remove_urls(text): return re.sub(r'http\S+|www\S+', '', text)
def remove_html(text): return BeautifulSoup(text, "html.parser").get_text()
def remove_emojis(text): return emoji.replace_emoji(text, replace='')

def replace_slang(text):
    pattern = r'\b(' + '|'.join(re.escape(w) for w in slang_dict.keys()) + r')\b'
    return re.sub(pattern, lambda m: slang_dict[m.group(0).lower()], text, flags=re.IGNORECASE)

def replace_contractions(text):
    pattern = r'\b(' + '|'.join(re.escape(w) for w in contractions_dict.keys()) + r')\b'
    return re.sub(pattern, lambda m: contractions_dict[m.group(0).lower()], text, flags=re.IGNORECASE)

def remove_punctuation(text): return text.translate(str.maketrans('', '', string.punctuation))
def remove_numbers(text): return re.sub(r'\d+', '', text)

def get_wordnet_pos(nltk_tag):
    if nltk_tag.startswith('J'): return wordnet.ADJ
    elif nltk_tag.startswith('V'): return wordnet.VERB
    elif nltk_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def lemmatize_text(text):
    if not isinstance(text, str): return ""
    words = word_tokenize(text)
    pos_tags = pos_tag(words)
    return " ".join([lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in pos_tags])

# Pipeline [cite: 479]
def preprocess_text(text):
    text = text.lower() # Step 1 [cite: 481]
    text = remove_urls(text) # Step 2 [cite: 482]
    text = remove_html(text) # Step 3 [cite: 483]
    text = remove_emojis(text) # Step 4 [cite: 484]
    text = replace_slang(text) # Step 5 [cite: 484]
    text = replace_contractions(text) # Step 6 [cite: 484]
    text = remove_punctuation(text) # Step 7 [cite: 484]
    text = remove_numbers(text) # Step 8 [cite: 484]
    text = spell(text) # Step 9 [cite: 484]
    text = " ".join([w for w in text.split() if w.lower() not in stop_words]) # Step 10 [cite: 484]
    text = lemmatize_text(text) # Step 11 [cite: 484]
    return word_tokenize(text) # Step 12 [cite: 485]

# Apply to Dataset [cite: 493, 495]
df = pd.read_csv("UNITENReview.csv")
df["processed"] = df["Review"].apply(preprocess_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
# Question 1
#the issues typically include:


#Inconsistent Casing: Mixed uppercase and lowercase letters.


#Web Noise: Presence of URLs and HTML tags.



#Informal Language: Use of emojis, internet slang (e.g., "tbh", "lol"), and contractions (e.g., "wasn't").



#Punctuation and Numbers: Extra exclamation marks, special characters, and numeric data.


#Noise Words: Common stopwords that do not contribute to the meaning.


#Word Variations: Different forms of the same root word (e.g., "running" vs "run")

In [5]:
#Question 2

# Load the exercise file
df_uniten = pd.read_csv("UNITENReview.csv")

# Apply the preprocessing pipeline you already created
# This handles lowercase, URLs, HTML, emojis, slang, contractions, 
# punctuation, numbers, spelling, stopwords, and lemmatization.
df_uniten["processed"] = df_uniten["Review"].apply(preprocess_text) 


In [7]:
# Question 3
# Save the result
df_uniten.to_csv("UNITEN_Processed_Result.csv", index=False)